In [72]:
#!git clone https://github.com/Felix-Schwer/rainfall_kf.git
%cd /content/rainfall_kf
!git pull origin main
!pip uninstall -y rainfall_kf
!pip install -e .

/content/rainfall_kf
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 2), reused 10 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.52 MiB | 6.46 MiB/s, done.
From https://github.com/Felix-Schwer/rainfall_kf
 * branch            main       -> FETCH_HEAD
   460188d..5e42b1c  main       -> origin/main
Updating 460188d..5e42b1c
Fast-forward
 data/gpm_sjv_subset.nc                             |  Bin 5055095 -> 0 bytes
 data/openet_sjv_subset.nc                          |  Bin 44717172 -> 0 bytes
 .../HUC8 - CONUS All.cpg}                          |    0
 data/shapefiles/HUC8 - CONUS All.dbf               |  Bin 0 -> 8581 bytes
 .../HUC8 - CONUS All.prj}                          |    0
 data/shapefiles/HUC8 - CONUS All.qmd               |  113 ++
 data/shapefiles/HUC8 - CONUS All.shp               |  Bin 0 -> 2085500 bytes
 data/shapefiles/HUC8 - CONUS All.shx     

In [79]:
import xarray as xr
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# PRISM

In [ ]:
gdf = gpd.read_file('/content/rainfall_kf/data/prism_sql_export.csv')

# OpenET

In [ ]:
da = xr.open_dataset('/content/rainfall_kf/data/openet_sjv_subset_huc8.nc')

<xarray.DataArray 'et' (time: 252, lat: 533, lon: 731)> Size: 393MB
[98184996 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 2kB 2000-01-01 2000-02-01 ... 2020-12-01
  * lat      (lat) float32 2kB 38.76 38.75 38.75 38.74 ... 36.38 36.37 36.37
  * lon      (lon) float32 3kB -121.9 -121.9 -121.9 ... -118.7 -118.7 -118.7
Attributes:
    units:      mm/month
    long_name:  monthly evapotranspiration (ensemble mean)

In [157]:
lon2d, lat2d = np.meshgrid(da.lon, da.lat)
points = gpd.GeoSeries(gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()), crs="EPSG:4326")

basins_df = gpd.read_file('/content/rainfall_kf/data/shapefiles/HUC8 - CONUS All.shp')

basin_ids = basins_df['HUC8']

et_means = []
for index, basin in basins_df.iterrows():
    basin = gpd.GeoDataFrame([basin], geometry='geometry', crs="EPSG:4326")
    inside = basin.union_all().contains(points).to_numpy().reshape(lat2d.shape)
    et_means.append(da.et.where(inside).mean(dim=['lon', 'lat']))
    print(f"Processed basin {basins_df.iloc[index]['NAME']} ({basin_ids[index]})")

mean_ds = xr.concat(et_means, dim=xr.DataArray(basin_ids, dims='basin', name='basin_HUC8'))
# create a dataset from the basin-mean DataArray and attach per-basin metadata as basin-coordinate variables
ds_et = xr.Dataset({"et": mean_ds})
ds_et.et.attrs['description'] = "Basin-averaged evapotranspiration from OpenET. Simple mean over all data points within each HUC8 basin."

# ensure basins are ordered to match the 'basin' coordinate in mean_ds
basins_by_huc = basins_df.set_index(basin_ids)
basins_ordered = basins_by_huc.loc[basin_ids]

# attach non-geometry metadata columns as basin coords
for col in basins_ordered.columns:
    if col == "geometry":
        continue
    ds_et = ds_et.assign_coords({f"basin_{col}": ("basin", basins_ordered[col].to_numpy())})

# attach geometry as WKT strings (safer for xarray coords)
ds_et = ds_et.assign_coords(
    {"basin_geometry_wkt": ("basin", basins_ordered.geometry.apply(lambda g: g.wkt).to_numpy())}
)

ds_et.attrs = da.attrs
ds_et.to_netcdf('/content/rainfall_kf/data/et_basin_means.nc')


Processed basin Middle San Joaquin-Lower Chowchilla (basin_ids[0])
Processed basin Lower San Joaquin River (basin_ids[1])
Processed basin San Joaquin Delta (basin_ids[2])
Processed basin Upper San Joaquin (basin_ids[3])
Processed basin Fresno River (basin_ids[4])
Processed basin Upper Merced (basin_ids[5])
Processed basin Upper Tuolumne (basin_ids[6])
Processed basin Upper Calaveras California (basin_ids[7])
Processed basin Panoche-San Luis Reservoir (basin_ids[8])
Processed basin Rock Creek-French Camp Slough (basin_ids[9])
Processed basin Upper Stanislaus (basin_ids[10])
Processed basin Upper Mokelumne (basin_ids[11])
Processed basin Upper Cosumnes (basin_ids[12])


# GPM

In [134]:
db = xr.open_dataset('/content/rainfall_kf/data/gpm_sjv_huc8_subset.nc')

In [ ]:
lon2d, lat2d = np.meshgrid(db.lon, db.lat)
points = gpd.GeoSeries(gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()), crs="EPSG:4326")

basins_df = gpd.read_file('/content/rainfall_kf/data/shapefiles/HUC8 - CONUS All.shp')

basin_ids = basins_df['HUC8']

precip_means = []
for index, basin in basins_df.iterrows():
    basin = gpd.GeoDataFrame([basin], geometry='geometry', crs="EPSG:4326")
    inside = basin.union_all().contains(points).to_numpy().reshape(lat2d.shape)
    precip_means.append(db.precipitation.where(inside).mean(dim=['lon', 'lat']))
    print(f"Processed basin {basins_df.iloc[index]['NAME']} ({basin_ids[index]})")

mean_ds = xr.concat(precip_means, dim=xr.DataArray(basin_ids, dims='basin', name='basin_HUC8'))
# create a dataset from the basin-mean DataArray and attach per-basin metadata as basin-coordinate variables
ds_precip = xr.Dataset({"precipitation": mean_ds})
ds_precip.precipitation.attrs['description'] = "Basin-averaged precipitation from GPM IMERG. Simple mean over all data points within each HUC8 basin."
# ensure basins are ordered to match the 'basin' coordinate in mean_ds
basins_by_huc = basins_df.set_index(basin_ids)
basins_ordered = basins_by_huc.loc[basin_ids]

# attach non-geometry metadata columns as basin coords
for col in basins_ordered.columns:
    if col == "geometry":
        continue
    ds_precip = ds_precip.assign_coords({f"basin_{col}": ("basin", basins_ordered[col].to_numpy())})

# attach geometry as WKT strings (safer for xarray coords)
ds_precip = ds_precip.assign_coords(
    {"basin_geometry_wkt": ("basin", basins_ordered.geometry.apply(lambda g: g.wkt).to_numpy())}
)

ds_precip.attrs = db.attrs
ds_precip.to_netcdf('/content/rainfall_kf/data/gpm_basin_means.nc')
